# Reinforcement Learning (DQN)

Compact notebook using the reusable `dqn` package modules.

## Setup

Run this cell once. In Colab it clones the repository and installs dependencies; locally it only adds `dqn/src` to the import path.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/miwehle/rl_lab.git"


def find_repo_root(start: Path) -> Path | None:
    for path in [start, *start.parents]:
        if (path / "dqn" / "src").exists():
            return path
    return None


try:
    import google.colab  # noqa: F401
    in_colab = True
except ImportError:
    in_colab = False

repo_root = find_repo_root(Path.cwd())

if repo_root is None and in_colab:
    if not Path("rl_lab").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("rl_lab")
    repo_root = Path.cwd()

if repo_root is None:
    raise RuntimeError("Run this notebook from the rl_lab repository or open it in Colab.")

if in_colab:
    os.chdir(repo_root)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "dqn/requirements.txt"], check=True)

src_dir = repo_root / "dqn" / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

repo_root

## Training

In [ ]:
import gymnasium as gym

from dqn.config import TrainingConfig
from dqn.training import Trainer
from dqn.visualize import EpisodePlotter

env = gym.make("CartPole-v1")

config = TrainingConfig(num_episodes=50)

plotter = EpisodePlotter(y_label="Duration")

trainer = Trainer(env, seed=42)

try:
    result = trainer.train(config, plotter=plotter)
finally:
    env.close()

plotter.plot_returns(result.episode_returns, show_result=True)

## Visualization

In [ ]:
from dqn.visualize import record_episode, show_animation

frames = record_episode(
    make_env=lambda: gym.make("CartPole-v1", render_mode="rgb_array"),
    q_net=result.q_net,
    device=trainer.device,
)

show_animation(frames)